# KYRBS 2023–2024: SAS → R analysis

This notebook reproduces the main steps in the supplied SAS program using R and the `survey` package.

**Flow:** import data → BMI classification → smartphone-use groups → covariates → missing-data exclusion → survey design → Table 1 (moonBook + Excel) → obesity prevalence (Table 2) → time-specific odds ratios.

The local SAS data files are expected in:
`C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data`

## Cell 1 — Install packages (run once)

If these packages are already installed, skip this cell.

In [ ]:
install.packages(c("haven", "dplyr", "tidyr", "survey", "broom", "moonBook", "openxlsx"))

## Cell 2 — Load packages

In [ ]:
library(haven)
library(dplyr)
library(tidyr)
library(survey)
library(broom)
library(moonBook)
library(openxlsx)

## Cell 3 — Set the data directory and check files

In [ ]:
DATA_DIR <- "C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data"

FILE_2023 <- file.path(DATA_DIR, "kyrbs2023.sas7bdat")
FILE_2024 <- file.path(DATA_DIR, "kyrbs2024.sas7bdat")

cat("2023 exists:", file.exists(FILE_2023), "\n")
cat("2024 exists:", file.exists(FILE_2024), "\n")

if (!file.exists(FILE_2023) || !file.exists(FILE_2024)) {
  stop("SAS files were not found. Check DATA_DIR and filenames.")
}

## Cell 4 — Read 2023 and 2024 SAS datasets and combine them

This corresponds to the SAS `data ky; set a.kyrbs2023 a.kyrbs2024; run;` step.

In [ ]:
DATA_DIR <- "C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data"

FILE_2023 <- file.path(DATA_DIR, "kyrbs2023.sas7bdat")
FILE_2024 <- file.path(DATA_DIR, "kyrbs2024.sas7bdat")

FILE_2023
FILE_2024

file.exists(FILE_2023)
file.exists(FILE_2024)

KY23 <- haven::read_sas(FILE_2023, encoding = "CP949")
KY24 <- haven::read_sas(FILE_2024, encoding = "CP949")

KY <- dplyr::bind_rows(KY23, KY24)

dim(KY)

In [ ]:
head(names(KY), 30)
stopifnot("HT" %in% names(KY), "WT" %in% names(KY))

## Cell 5 — Calculate BMI

In [ ]:
A2 <- KY %>%
  mutate(
    K = HT / 100,
    BMI = round(WT / (K * K), 9)
  )

## Cell 6 — Assign sex/age-specific BMI percentile cutoffs

In [ ]:
A2 <- A2 %>%
  mutate(
    PCT05 = case_when(
      SEX == 1 & AGE_M == 144 ~ 15.5,
      SEX == 1 & AGE_M == 145 ~ 15.6,
      SEX == 1 & AGE_M == 146 ~ 15.6,
      SEX == 1 & AGE_M >= 147 & AGE_M <= 227 ~ 17.0,
      SEX == 2 & AGE_M == 144 ~ 15.3,
      SEX == 2 & AGE_M == 145 ~ 15.3,
      SEX == 2 & AGE_M == 146 ~ 15.4,
      SEX == 2 & AGE_M >= 147 & AGE_M <= 227 ~ 16.5,
      TRUE ~ NA_real_
    ),
    PCT85 = case_when(
      SEX == 1 & AGE_M == 144 ~ 23.0,
      SEX == 1 & AGE_M == 145 ~ 23.0,
      SEX == 1 & AGE_M == 146 ~ 23.1,
      SEX == 1 & AGE_M >= 147 & AGE_M <= 227 ~ 24.5,
      SEX == 2 & AGE_M == 144 ~ 22.1,
      SEX == 2 & AGE_M == 145 ~ 22.2,
      SEX == 2 & AGE_M == 146 ~ 22.2,
      SEX == 2 & AGE_M >= 147 & AGE_M <= 227 ~ 23.5,
      TRUE ~ NA_real_
    ),
    PCT95 = case_when(
      SEX == 1 & AGE_M == 144 ~ 25.1,
      SEX == 1 & AGE_M == 145 ~ 25.1,
      SEX == 1 & AGE_M == 146 ~ 25.2,
      SEX == 1 & AGE_M >= 147 & AGE_M <= 227 ~ 26.5,
      SEX == 2 & AGE_M == 144 ~ 24.1,
      SEX == 2 & AGE_M == 145 ~ 24.2,
      SEX == 2 & AGE_M == 146 ~ 24.2,
      SEX == 2 & AGE_M >= 147 & AGE_M <= 227 ~ 25.5,
      TRUE ~ NA_real_
    )
  )

## Cell 7 — Create BMI groups (`g_bmi`)

In [ ]:
A3 <- A2 %>%
  mutate(
    G_BMI = case_when(
      !is.na(BMI) & !is.na(PCT05) & !is.na(PCT85) & !is.na(PCT95) & BMI >= PCT95 ~ 4,
      !is.na(BMI) & !is.na(PCT05) & !is.na(PCT85) & !is.na(PCT95) & BMI >= PCT85 & BMI < PCT95 ~ 3,
      !is.na(BMI) & !is.na(PCT05) & !is.na(PCT85) & !is.na(PCT95) & BMI >= PCT05 & BMI < PCT85 ~ 2,
      !is.na(BMI) & !is.na(PCT05) & !is.na(PCT85) & !is.na(PCT95) & BMI < PCT05 ~ 1,
      TRUE ~ NA_real_
    )
  )

## Cell 8 — Calculate average daily smartphone use

In [ ]:
A4 <- A3 %>%
  mutate(
    SP_WD_HR = INT_SPWD_TM / 60,
    SP_WK_HR = INT_SPWK_TM / 60,
    SP_AVG = (SP_WD_HR * 5 + SP_WK_HR * 2) / 7
  )

## Cell 9 — Create smartphone-use groups (`time`)

In [ ]:
A4 <- A4 %>%
  mutate(
    TIME = case_when(
      SP_AVG < 2 ~ 1,
      SP_AVG >= 2 & SP_AVG < 4 ~ 2,
      SP_AVG >= 4 & SP_AVG < 6 ~ 3,
      SP_AVG >= 6 ~ 4,
      TRUE ~ NA_real_
    )
  )

## Cell 10 — Create region, smoking, education, economic status, stress, depression, drinking, and age group

In [ ]:
A4 <- A4 %>%
  mutate(
    REGION = case_when(
      as.character(CTYPE) == "군지역" ~ 2,
      as.character(CTYPE) %in% c("대도시", "중소도시") ~ 1,
      TRUE ~ NA_real_
    ),

    SMOKING1 = if_else(TC_DAYS %in% c(1, 9999) | is.na(TC_DAYS), 0, 1),
    SMOKING2 = if_else(TC_EC_MN %in% c(1, 9999) | is.na(TC_EC_MN), 0, 1),
    SMOKING3 = if_else(TC_HTP_MN %in% c(1, 9999) | is.na(TC_HTP_MN), 0, 1),
    SMOKING = if_else(SMOKING1 == 1 | SMOKING2 == 1 | SMOKING3 == 1, 1, 0),

    EDU = case_when(
      E_S_RCRD %in% c(1, 2) ~ 1,
      E_S_RCRD == 3 ~ 2,
      E_S_RCRD %in% c(4, 5) ~ 3,
      TRUE ~ NA_real_
    ),

    ECO = case_when(
      E_SES %in% c(1, 2) ~ 1,
      E_SES == 3 ~ 2,
      E_SES %in% c(4, 5) ~ 3,
      TRUE ~ NA_real_
    ),

    STRESS = if_else(M_STR %in% c(1, 2), 1, 0),
    DEPRESS = if_else(M_SAD == 2, 1, 0),
    DRINKING = if_else(AC_DAYS %in% c(1, 9999) | is.na(AC_DAYS), 0, 1),

    AGE_G = case_when(
      AGE %in% c(12, 13, 14) ~ 1,
      AGE %in% c(15, 16, 17, 18) ~ 2,
      TRUE ~ NA_real_
    )
  ) %>%
  filter(YEAR %in% c(2023, 2024))

## Cell 11 — Keep the same analysis variables as the SAS program

In [ ]:
A4 <- A4 %>%
  select(
    YEAR, AGE_G, SEX, REGION, G_BMI, TIME,
    SMOKING, DRINKING, EDU, ECO, STRESS, DEPRESS,
    W, CLUSTER, STRATA
  )

## Cell 12 — Check frequencies before deleting missing observations

In [ ]:
freq_vars <- c("YEAR", "SEX", "AGE_G", "REGION", "G_BMI", "EDU", "ECO",
               "SMOKING", "DRINKING", "TIME", "STRESS", "DEPRESS")

for (v in freq_vars) {
  cat("\n====================", v, "====================\n")
  print(table(A4[[v]], useNA = "ifany"))
}

## Cell 13 — Exclude records with missing age group, BMI group, education, or economic status

In [ ]:
A5 <- A4 %>%
  filter(
    !is.na(AGE_G),
    !is.na(G_BMI),
    !is.na(EDU),
    !is.na(ECO)
  )

## Cell 14 — Create the binary obesity outcome

In [ ]:
A5 <- A5 %>%
  mutate(
    OBESE = case_when(
      G_BMI %in% c(1, 2, 3) ~ 0,
      G_BMI == 4 ~ 1,
      TRUE ~ NA_real_
    )
  )

## Cell 15 — Check the final analytic sample

In [ ]:
cat("Final N:", nrow(A5), "\n")

for (v in c(freq_vars, "OBESE")) {
  cat("\n====================", v, "====================\n")
  print(table(A5[[v]], useNA = "ifany"))
}

## Cell 16 — Define the complex survey design

In [ ]:
options(survey.lonely.psu = "adjust")

DESIGN <- svydesign(
  ids = ~CLUSTER,
  STRATA = ~STRATA,
  weights = ~W,
  data = A5,
  nest = TRUE
)

DESIGN

## Cell 17 — Table 1: create the table with `moonBook`

In [ ]:
A5_labeled <- A5 %>%
  mutate(
    TIME = factor(TIME, levels = 1:4, labels = c("<2h", "2h–4h", "4h–6h", ">6h")),
    SEX = factor(SEX, levels = c(1, 2), labels = c("Male", "Female")),
    AGE_G = factor(AGE_G, levels = c(1, 2), labels = c("7th–9th grade", "10th–12th grade")),
    REGION = factor(REGION, levels = c(1, 2), labels = c("Urban", "Rural")),
    G_BMI = factor(G_BMI, levels = c(1, 2, 3, 4),
                   labels = c("Underweight", "Normal", "Overweight", "Obese")),
    EDU = factor(EDU, levels = c(1, 2, 3), labels = c("High", "Middle", "Low")),
    ECO = factor(ECO, levels = c(1, 2, 3), labels = c("High", "Middle", "Low")),
    STRESS = factor(STRESS, levels = c(0, 1), labels = c("Low", "High")),
    DEPRESS = factor(DEPRESS, levels = c(0, 1), labels = c("Low", "High")),
    SMOKING = factor(SMOKING, levels = c(0, 1), labels = c("Non-smoker", "Smoker")),
    DRINKING = factor(DRINKING, levels = c(0, 1), labels = c("Non-drinker", "Drinker"))
  )

moonbook_table1 <- mytable(
  TIME ~ SEX + AGE_G + REGION + G_BMI + EDU + ECO + STRESS + DEPRESS +
    SMOKING + DRINKING,
  data = A5_labeled,
  show.total = TRUE,
  show.all = FALSE
)

moonbook_table1

## Cell 18 — Table 1: make the manuscript-style Excel table

In [ ]:
TABLE1_VARS <- c(
  SEX = "Sex",
  AGE_G = "Grade",
  REGION = "Region of residence",
  G_BMI = "BMI*",
  EDU = "Academic achievement",
  ECO = "Economic level",
  STRESS = "Stress",
  DEPRESS = "Depression",
  SMOKING = "Smoking status",
  DRINKING = "Alcohol consumption"
)

TIME_LEVELS <- c("<2h", "2h–4h", "4h–6h", ">6h")

make_table1 <- function(data, variables, time_var = "TIME") {
  data[[time_var]] <- as.character(data[[time_var]])
  total_n <- nrow(data)
  result <- list()
  k <- 1

  # Overall row
  overall_row <- data.frame(
    Characteristics = "Overall",
    Total = sprintf("%d", total_n),
    `<2h` = sprintf("%d (%.2f)", sum(!is.na(data[[time_var]]) & data[[time_var]] == "<2h"),
                    mean(!is.na(data[[time_var]]) & data[[time_var]] == "<2h") * 100),
    `2h–4h` = sprintf("%d (%.2f)", sum(!is.na(data[[time_var]]) & data[[time_var]] == "2h–4h"),
                      mean(!is.na(data[[time_var]]) & data[[time_var]] == "2h–4h") * 100),
    `4h–6h` = sprintf("%d (%.2f)", sum(!is.na(data[[time_var]]) & data[[time_var]] == "4h–6h"),
                      mean(!is.na(data[[time_var]]) & data[[time_var]] == "4h–6h") * 100),
    `>6h` = sprintf("%d (%.2f)", sum(!is.na(data[[time_var]]) & data[[time_var]] == ">6h"),
                    mean(!is.na(data[[time_var]]) & data[[time_var]] == ">6h") * 100),
    check.names = FALSE,
    stringsAsFactors = FALSE
  )
  result[[k]] <- overall_row
  k <- k + 1

  for (v in names(variables)) {
    # Section/header row
    result[[k]] <- data.frame(
      Characteristics = variables[[v]],
      Total = "",
      `<2h` = "",
      `2h–4h` = "",
      `4h–6h` = "",
      `>6h` = "",
      check.names = FALSE,
      stringsAsFactors = FALSE
    )
    k <- k + 1

    levs <- levels(factor(data[[v]]))

    for (lev in levs) {
      total_count <- sum(!is.na(data[[v]]) & data[[v]] == lev)
      total_pct <- ifelse(total_n > 0, total_count / total_n * 100, NA_real_)

      row <- data.frame(
        Characteristics = paste0("  ", lev),
        Total = sprintf("%d (%.2f)", total_count, total_pct),
        `<2h` = "",
        `2h–4h` = "",
        `4h–6h` = "",
        `>6h` = "",
        check.names = FALSE,
        stringsAsFactors = FALSE
      )

      for (tt in TIME_LEVELS) {
        d_group <- data[!is.na(data[[time_var]]) & data[[time_var]] == tt & !is.na(data[[v]]), , drop = FALSE]
        n_group <- nrow(d_group)
        n_level <- sum(d_group[[v]] == lev)
        pct_group <- ifelse(n_group > 0, n_level / n_group * 100, NA_real_)
        row[[tt]] <- sprintf("%d (%.2f)", n_level, pct_group)
      }

      result[[k]] <- row
      k <- k + 1
    }
  }

  bind_rows(result)
}

Table1 <- make_table1(A5_labeled, TABLE1_VARS)
Table1

## Cell 18 Export — Export Table 1 to Excel

In [ ]:
TABLE1_XLSX <- file.path(DATA_DIR, "Table1_screen_time.xlsx")

wb <- createWorkbook()
addWorksheet(wb, "Table 1")

writeData(wb, "Table 1", Table1, startRow = 1, startCol = 1, rowNames = FALSE)

header_style <- createStyle(textDecoration = "bold", halign = "center", border = "Bottom")
section_style <- createStyle(textDecoration = "bold")

addStyle(wb, "Table 1", header_style,
         rows = 1, cols = 1:ncol(Table1), gridExpand = TRUE)

section_rows <- which(Table1$Total == "" & Table1$Characteristics != "Overall") + 1
if (length(section_rows) > 0) {
  addStyle(wb, "Table 1", section_style,
           rows = section_rows, cols = 1, gridExpand = FALSE)
}

setColWidths(wb, "Table 1", cols = 1, widths = 30)
setColWidths(wb, "Table 1", cols = 2:ncol(Table1), widths = 18)
freezePane(wb, "Table 1", firstRow = TRUE)

saveWorkbook(wb, TABLE1_XLSX, overwrite = TRUE)